# langchain core
랭체인 기본 함수들 예시
- PromptTemplate
- RunnableLambda, RunnableBranch
- PydanticOutputParser
- StateGraph
- MemorySaver

In [47]:
import json
from pydantic import BaseModel, Field
from typing import Annotated
from typing_extensions import TypedDict

from openai import AsyncOpenAI

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.runnables import RunnableLambda, RunnableBranch


from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

from langgraph.graph.state import CompiledStateGraph


# Model

In [20]:
## Init Model
# Model Init
LLM_BASE_URL = "http://localhost:901/v1"
LLM_API_KEY = "sk-123"
LLM_MODEL = "Qwen3.5-35B-A3B"
# LLM_MODEL = "Qwen3.5-122B-A10B"

model = ChatOpenAI(
    model=LLM_MODEL,
    base_url=LLM_BASE_URL,
    api_key=LLM_API_KEY,
    # async_client=AsyncOpenAI(
    #     base_url=LLM_BASE_URL,
    #     api_key=LLM_API_KEY,
    # ).chat.completions,
)

In [21]:
# Normal Invoke
await model.ainvoke("Hello", temperature=0.7)

AIMessage(content='Hello! How can I help you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 13, 'total_tokens': 23, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Qwen3.5-35B-A3B', 'system_fingerprint': None, 'id': 'chatcmpl-b77ab84aa64477da', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cf091-afcb-7173-9f6b-9867e0058fcf-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 10, 'total_tokens': 23, 'input_token_details': {}, 'output_token_details': {}})

In [22]:
# Normal Invoke
messages = [HumanMessage(content="What is the capital of France?")]
await model.ainvoke(messages, temperature=0.7)

AIMessage(content="The capital of France is **Paris**.\n\nLocated in the north-central part of the country along the Seine River, Paris is the nation's foremost center of finance, commerce, culture, arts, fashion, and science. It has been the capital for centuries and remains one of the most populous and influential cities in Europe.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 65, 'prompt_tokens': 19, 'total_tokens': 84, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Qwen3.5-35B-A3B', 'system_fingerprint': None, 'id': 'chatcmpl-844d269db87d2281', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cf091-b16d-7f11-9a3c-1625887bdf34-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 19, 'output_tokens': 65, 'total_tokens': 84, 'input_token_details': {}, 'output_token_details': {}})

In [23]:
# Structured Output
class Answer(BaseModel):
    city: str
    country: str

structured_model = model.with_structured_output(Answer)

response = await structured_model.ainvoke("What is the capital of France?")
print(response.city)     # "Paris"
print(response.country) 

Paris
France


/home/yrlab/miniconda3/envs/agent/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=Answer(city='Paris', country='France'), input_type=Answer])
  return self.__pydantic_serializer__.to_python(


In [26]:
response = await model.ainvoke("Hello", temperature=0.7, response_format=Answer)
json.loads(response.content)

/home/yrlab/miniconda3/envs/agent/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=Answer(city='Beijing', country='China'), input_type=Answer])
  return self.__pydantic_serializer__.to_python(


{'city': 'Beijing', 'country': 'China'}

# RunnableLambda

In [36]:
CHATBOT_INSTRUCTION='''You are a helpful chatbot.
Query: {query}'''
async def chatbot(inputs):
    query = inputs["query"]
    response = await model.ainvoke(CHATBOT_INSTRUCTION.format(query=query), temperature=0.3)
    inputs["response"] = response.content
    return inputs

query = "What is the capital of France?"
response = await RunnableLambda(chatbot).ainvoke({"query": query})

In [37]:
print(response)

{'query': 'What is the capital of France?', 'response': 'The capital of France is **Paris**.'}


In [40]:
TRANSLATE_INSTRUCTION='''Translate this text to {lang}
Text: {text}'''

async def translate(inputs):
    text = inputs["response"]
    lang = inputs["lang"]
    response = await model.ainvoke(
        TRANSLATE_INSTRUCTION.format(
            text=text,
            lang=lang
        ),
        temperature=0.3
    )
    inputs["translation"] = response.content
    return inputs

chatbot_lambda = RunnableLambda(chatbot)
translate_lambda = RunnableLambda(translate)
chain = chatbot_lambda | translate_lambda

response = await chain.ainvoke({"query": query, "lang": "ko"})

In [41]:
print(response["translation"])

프랑스의 수도는 **파리**입니다.


# RunnableBranch

In [44]:
TRANSLATE_KO_INSTRUCTION='''Translate this text to korean
Text: {text}'''

TRANSLATE_JP_INSTRUCTION='''Translate this text to japanese
Text: {text}'''

async def translate_ko(inputs):
    text = inputs["response"]
    response = await model.ainvoke(
        TRANSLATE_KO_INSTRUCTION.format(
            text=text,
        ),
        temperature=0.3
    )
    inputs["translation"] = response.content
    return inputs

async def translate_jp(inputs):
    text = inputs["response"]
    response = await model.ainvoke(
        TRANSLATE_JP_INSTRUCTION.format(
            text=text,
        ),
        temperature=0.3
    )
    inputs["translation"] = response.content
    return inputs


# Branch
translate_branch = RunnableBranch(
    (lambda x: x["lang"]=="ko", translate_ko),
    translate_jp
)

chain = chatbot_lambda | translate_branch

response = await chain.ainvoke({"query": query, "lang": "jp"})

In [45]:
print(response["translation"])

フランスの首都は**パリ**です。


# StateGraph

In [48]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.state import CompiledStateGraph
from langgraph.graph.message import add_messages

In [50]:
# 1. State 정의
class State(TypedDict):
    messages: Annotated[list, add_messages]  # 대화 히스토리 누적

# 2. 챗봇 노드
async def chatbot(state: State):
    response = await model.ainvoke(state["messages"])
    return {"messages": [response]}

# 3. 그래프 구성
builder = StateGraph(State)
builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

graph = builder.compile()


In [51]:
response = await graph.ainvoke(
    {"messages": [HumanMessage(content="내 이름은 철수야")]},
)
print(response["messages"][-1].content)

안녕하세요, 철수님! 반갑습니다. 오늘 어떤 도움이 필요하신가요? 궁금한 점이 있거나 대화하고 싶은 주제가 있다면 언제든 말씀해 주세요! 😊


# Memory

## 개념
Threads
- A thread is a unique ID or thread identifier assigned to each checkpoint saved by a checkpointer. It contains the accumulated state of a sequence of runs. When a run is executed, the state of the underlying graph of the assistant will be persisted to the thread.
- When invoking a graph with a checkpointer, you must specify a thread_id as part of the configurable portion of the config:
    - {"configurable": {"thread_id": "1"}}


참고 자료
- https://docs.langchain.com/oss/python/langgraph/persistence

In [57]:
from langgraph.checkpoint.memory import MemorySaver

In [58]:
memory = MemorySaver()
graph = builder.compile(checkpointer=memory)

In [59]:
# 5. thread_id로 대화 세션 구분
config = {"configurable": {"thread_id": "user_123"}}

# 첫 번째 메시지
response = await graph.ainvoke(
    {"messages": [HumanMessage(content="내 이름은 철수야")]},
    config=config
)
print(response["messages"][-1].content)
# "안녕하세요, 철수님! ..."

반갑습니다, 철수님! 👋  
무엇을 도와드릴까요? 궁금한 점이 있거나 대화하고 싶은 주제가 있다면 언제든지 말씀해 주세요.


In [60]:
# 두 번째 메시지 — 이전 대화를 기억함
response = await graph.ainvoke(
    {"messages": [HumanMessage(content="내 이름이 뭐야?")]},
    config=config
)
print(response["messages"][-1].content)

방금 말씀해 주셨잖아요, 철수님! 😄  
무엇을 도와드릴까요?
